In [23]:
import requests
import json
import pandas as pd
from prefect import flow, task
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check
from etl.common.loaders import load_valid_country_ids


# ======================
# Definición de esquema
# ======================

gini_schema = DataFrameSchema({
    "cca3": Column(
        str,
        [
            Check.str_length(3, 3),
            Check.isin(load_valid_country_ids())
        ],
        nullable=False
    ),
    "year": Column(int, [
        Check.ge(1900),
        Check.le(2100)
    ]),
    "gini": Column(float, [
        Check.ge(0),
        Check.le(100)
    ])
})


# ======================
# Tareas
# ======================

@task
def extract_data():
    url = "https://restcountries.com/v3.1/all?fields=cca3,gini"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        print(f"Se cargaron {len(data)} países")
        print(json.dumps(data[0], indent=4, ensure_ascii=False))
        return data
    else:
        raise Exception(f"Error al conectar con la API: {response.status_code}")


@task
def transform_data(data):
    df = pd.DataFrame(data)

    def parse_gini(entry):
        if isinstance(entry, dict) and entry:
            years = list(entry.keys())
            values = [
                float(v) if isinstance(v, (int, float, str)) and v not in ("", None) else None
                for v in entry.values()
            ]
            return years, values
        return [], []

    df["years"], df["values"] = zip(*df["gini"].apply(parse_gini))
    df = df.explode(["years", "values"], ignore_index=True)

    df = df.dropna(subset=["values", "years"])
    df["year"] = df["years"].astype(int)
    df["gini"] = df["values"].astype(float)

    df = df[["cca3", "year", "gini"]]
    return df


@task
def validate_data(df: pd.DataFrame) -> pd.DataFrame:
    return gini_schema.validate(df)


@task
def load_to_csv(df, path="./stage/country_gini.csv"):
    df.to_csv(path, index=False)
    print(f"Archivo guardado en {path}")


# ======================
# Flow principal
# ======================

@flow(name="etl_gini_flow")
def etl_gini_flow():
    data = extract_data()
    df = transform_data(data)
    df = validate_data(df)  # validación
    load_to_csv(df)


if __name__ == "__main__":
    etl_gini_flow()


04:08:25.383 | INFO    | Flow run 'divergent-mouse' - Beginning flow run 'divergent-mouse' for flow 'etl_gini_flow'

Se cargaron 250 países
{
    "cca3": "JAM",
    "gini": {
        "2004": 45.5
    }
}


04:08:26.361 | INFO    | Task run 'extract_data-e8d' - Finished in state Completed()

04:08:26.597 | INFO    | Task run 'transform_data-915' - Finished in state Completed()

04:08:26.819 | INFO    | Task run 'validate_data-253' - Finished in state Completed()

Archivo guardado en ./stage/country_gini.csv


04:08:27.040 | INFO    | Task run 'load_to_csv-45b' - Finished in state Completed()

04:08:27.052 | INFO    | Flow run 'divergent-mouse' - Finished in state Completed()